In [1]:
# Imports
import sys
import pandas as pd

import os
import json 
from pathlib import Path

# Access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from app.services.quiz_router import handle_quiz_request
from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import Dict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from app.agent.rag.chains.eval_chain import run_eval_chain


In [4]:

load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",  # The current efficient workhorse model
    project="denseless",       # Replace with your actual Project ID
    location="us-central1",              # Or your preferred region
    vertexai=True,
)

json_eval_dataset = {}
quizzes: Dict[str, List] = {}

INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [5]:

topics = ['ADDER CIRCUIT.pdf', 'Disaster Recovery Plan.pdf', 'Incident Response.pdf', 'PROCESS MANAGEMENT.pdf', 'Software Engineering Intro.pdf']

# Add all topics to store first - Added
# for topic in topics:
#     topic_name = topic.removesuffix('.pdf')
#     docs = process_and_load_file(f"../pdfs/{topic}")
#     embedder = get_embedding_model()
#     vectors = generate_embeddings(docs, embedder)
#     store = get_vector_store(student_id="1019", embedder=embedder)
#     add_documents_to_chroma(store, vectors, docs, False, "FQ", topic_name, topic_name)


embedder = get_embedding_model()
store = get_vector_store(student_id="1019", embedder=embedder)
embedder = get_embedding_model()

# Generate 10 quizzes for each topic
for topic in topics:
    topic_name = topic.removesuffix('.pdf')
    quizzes[topic] = []

    for _ in range(10):
        result = handle_quiz_request(
            student_id      = "student_1019",
            course          = "FQ",
            topic           = topic_name,
            store           = store,
            llm             = llm,
            simulated_today = "2026-05-11"
        )
        print(result["status"])
        quiz_file_name = result["saved_path"].removeprefix("data/quizzes/")
        quizzes[topic].append(quiz_file_name)


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BA

...

[quiz_chain] JsonOutputParser succeeded.
[quiz_chain] Running per-question source mapping...
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What are the two main components that software comprises, be'
Querying Chroma (top_k=3, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 3 chunk(s), 3 passed threshold.
[Retriever] → 3 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 2 section(s): {'The Term Software', 'Today’s software is comprised of'}
[Retriever]   → Section 'The Term Software': 2 chunk(s) fetched.
[Retriever]   → Section 'Today’s software is comprised of': 13 chunk(s) fetched.
[Retriever] → Expansion complete: 3 seed → 15 unique chunk(s) after deduplication.
[Retriever] → Final result: 15 chunk(s) returned to chain.
[quiz_chain] Q1 mapped — pages: [1, 2], section: The Term Software
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4):

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  → Retrieved 3 chunk(s), 3 passed threshold.
[Retriever] → 3 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 2 section(s): {'OR', 'The Term Software Engineering'}
[Retriever]   → Section 'OR': 3 chunk(s) fetched.
[Retriever]   → Section 'The Term Software Engineering': 6 chunk(s) fetched.
[Retriever] → Expansion complete: 3 seed → 9 unique chunk(s) after deduplication.
[Retriever] → Final result: 9 chunk(s) returned to chain.
[quiz_chain] Q10 mapped — pages: [3, 4], section: The Term Software Engineering
[quiz_chain] Complete — questions: 10/10, tokens: 4041
[quiz_chain] Quiz saved to: c:\Users\USER\Documents\AI projects\DenseLess\data\quizzes\student_1019_software_engineering_intro_20260525_122002.json
  [token_service] Deducted 4,041 tokens — student: 'student_1019' | chain: run_quiz_chain | in: 2,797 | out: 1,244 | remaining: 7,483,984
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 4041 |

INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[quiz_chain] JsonOutputParser succeeded.
[quiz_chain] Running per-question source mapping...
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What are the two main components that software comprises, be'
Querying Chroma (top_k=3, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 3 chunk(s), 3 passed threshold.
[Retriever] → 3 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 2 section(s): {'The Term Software', 'Today’s software is comprised of'}
[Retriever]   → Section 'The Term Software': 2 chunk(s) fetched.
[Retriever]   → Section 'Today’s software is comprised of': 13 chunk(s) fetched.
[Retriever] → Expansion complete: 3 seed → 15 unique chunk(s) after deduplication.
[Retriever] → Final result: 15 chunk(s) returned to chain.
[quiz_chain] Q1 mapped — pages: [1, 2], section: The Term Software
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4):

In [ ]:
# Answer all quizzes manually

In [ ]:
# Evaluate each quiz and store score and feedback
i = 1
for topic in topics:
    topic_name = topic.removesuffix('.pdf')

    for quiz_file_name in quizzes[topic]:
        response = run_eval_chain(
            student_id     = "student_1019",
            topic          = topic_name,
            quiz_phase     = "revision",
            quiz_path      = f"../../data/quizzes/{quiz_file_name}",
            llm            = llm,
            simulated_date = "2026-05-11"
        )
        feedback = response.content["feedback"]
        score = response.content["questions"][0]["score"] * 100

        key = str(i).zfill(3)
        json_eval_dataset[key] = {
            "topic_title"      : topic_name,
            "quiz_file"        : quiz_file_name,
            "overall_quiz_score": score,
            "feedback_given"   : feedback,
        }
        i += 1


In [ ]:
# Export to a json
output_path = Path(os.getcwd()) / "rag" / "eval_data" / "fq_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(json_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"Successfully completed. Exported to {output_path}")